In [1]:
pip install torch transformers scikit-learn matplotlib seaborn pandas tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
import json
import numpy as np
from sklearn.cluster import KMeans
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import os
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import warnings
from itertools import permutations

/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/LSD-Analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [4]:
# Load Pre-computed SinLlama Contextual Embeddings
print("Loading embeddings...")
century_embeddings = torch.load("../Data/XLM-R_All_Century_Embeddings.pt", weights_only=False)

# Filter Centuries (Keep only robust ones)
valid_centuries = ['13', '14', '15', '18', '19', '20']
century_embeddings = {k: v for k, v in century_embeddings.items() if k in valid_centuries}
print(f"Loaded centuries: {list(century_embeddings.keys())}")

Loading embeddings...
Loaded centuries: ['13', '14', '15', '18', '19', '20']


In [5]:
# Load Lemma Data for Aggregation
print("Loading lemma and token data...")
with open('../Data/century_lemmas.json', 'r', encoding='utf-8') as f:
    century_lemmas = json.load(f)

with open('../Data/century_tokens.json', 'r', encoding='utf-8') as f:
    century_tokens = json.load(f)

# Build Sentence-Aligned Lemma Map
# Structure: lemma_map[century][sentence_index][word_index] = lemma
lemma_lookup = defaultdict(lambda: defaultdict(dict))

for century in valid_centuries:
    if century not in century_lemmas: continue
    l_sentences = century_lemmas[century]
    
    for s_idx, l_sent in enumerate(l_sentences):
        for w_idx, lemma in enumerate(l_sent):
            if lemma:
                lemma_lookup[century][s_idx][w_idx] = lemma
print("Lemma lookup built.")

Loading lemma and token data...
Lemma lookup built.


In [6]:
lemma_lookup

defaultdict(<function __main__.<lambda>()>,
            {'13': defaultdict(dict,
                         {0: {0: 'අප',
                           1: 'බුදුන්',
                           2: 'සාරාස',
                           3: 'කප්සුවහස',
                           4: 'මතු',
                           5: 'කුලුණුනු',
                           6: 'වණ',
                           7: 'යුත්',
                           8: 'මහත',
                           9: 'ව',
                           10: 'සක්',
                           11: 'වැ',
                           12: 'දිවකුරු',
                           13: 'බුදුන්',
                           14: 'හමුව',
                           15: 'අත',
                           16: 'ප',
                           17: 'නිවන්',
                           18: 'සසර',
                           19: 'සේ',
                           20: 'පි',
                           21: 'සත',
                           22: 'සඳහා',
                           23: 'විඳි

In [7]:
lemma_rows = []

for century, sentences in tqdm(century_embeddings.items(), desc="Processing Instances"):
    # Get the list of original text for this century
    # Replace 'all_original_texts' with your actual text variable
    raw_texts = century_tokens.get(century, []) 

    for s_idx, sentence_data in enumerate(sentences):
        # Capture the full sentence text for this index
        full_sentence_text = raw_texts[s_idx] if s_idx < len(raw_texts) else "Text not found"
        
        for w_idx, (word, vector) in enumerate(sentence_data):
            lemma = lemma_lookup[century].get(s_idx, {}).get(w_idx)
            
            if lemma:
                lemma_rows.append({
                    "century": int(century), # Convert to int for easier filtering
                    "word": word,
                    "lemma": lemma,
                    "vector": vector.numpy(),
                    "sentence_text": full_sentence_text  # <--- CRITICAL ADDITION
                })

df_results = pd.DataFrame(lemma_rows)

Processing Instances: 100%|██████████| 6/6 [00:00<00:00, 16.99it/s]


In [8]:
df_results = pd.DataFrame(lemma_rows)

In [9]:
df_results.head()

,century,word,lemma,vector,sentence_text
0,13,අප,අප,"[-0.017851107, -0.044579446, -0.040111683, 0.0...","[අප, බුදුන්, සාරාසකි, කප්සුවහස්, මතුයෙහි, කුලු..."
1,13,බුදුන්,බුදුන්,"[0.041268058, -0.035116285, -0.028367188, 0.00...","[අප, බුදුන්, සාරාසකි, කප්සුවහස්, මතුයෙහි, කුලු..."
2,13,සාරාසකි,සාරාස,"[-0.018791229, 0.004046296, -0.040097833, 0.03...","[අප, බුදුන්, සාරාසකි, කප්සුවහස්, මතුයෙහි, කුලු..."
3,13,කප්සුවහස්,කප්සුවහස,"[-0.014488558, 0.00863839, -0.033872638, 0.037...","[අප, බුදුන්, සාරාසකි, කප්සුවහස්, මතුයෙහි, කුලු..."
4,13,මතුයෙහි,මතු,"[0.030637398, -0.0062566586, -0.05201265, 0.04...","[අප, බුදුන්, සාරාසකි, කප්සුවහස්, මතුයෙහි, කුලු..."


In [10]:
def get_lemma_map(sub_df):
    l_map = {}
    for idx, row in sub_df.iterrows():
        l = row['lemma']
        
        # 1. THE FIX: Initialize the 'texts' list here
        if l not in l_map: 
            l_map[l] = {
                'vecs': [], 
                'indices': [], 
                'texts': []  # <--- MUST ADD THIS LINE
            }
        
        l_map[l]['vecs'].append(np.array(row['vector']))
        l_map[l]['indices'].append(idx)
        
        # 2. Now this line will work because the key exists
        sentence_string = " ".join(row['sentence_text']) if isinstance(row['sentence_text'], list) else row['sentence_text']
        l_map[l]['texts'].append(sentence_string)
        
    return l_map

In [11]:
# # 3. Define the LOO Logic
# def run_loo(l_old, l_new):
#     v_old = np.array(l_old['vecs'])
#     v_new = np.array(l_new['vecs'])
#     ids_new = l_new['indices']
#     texts_new = l_new['texts']
    
#     c_old = np.mean(v_old, axis=0)
#     c_new = np.mean(v_new, axis=0)
    
#     # Standard Drift (Distance between full averages)
#     base_dist = 1 - np.clip(cosine_similarity(c_old.reshape(1,-1), c_new.reshape(1,-1))[0][0], -1, 1)
    
#     loo_dists = []
#     impacts = []
    
#     for j in range(len(v_new)):
#         mask = np.ones(len(v_new), dtype=bool)
#         mask[j] = False
#         if np.any(mask):
#             c_loo = np.mean(v_new[mask], axis=0)
#             d_loo = 1 - np.clip(cosine_similarity(c_old.reshape(1,-1), c_loo.reshape(1,-1))[0][0], -1, 1)
#             loo_dists.append(d_loo)
#             impacts.append(base_dist - d_loo)
#         else:
#             loo_dists.append(base_dist)
#             impacts.append(0)

#     best_index = np.argmax(impacts)
    
#     return {
#         'persistent_shift': min(loo_dists),
#         'global_distance': base_dist,
#         'innovation_instances': texts_new[best_index], # Now texts_new is defined!
#         'max_impact': max(impacts) if impacts else 0
#     }

In [12]:
def run_multi_loo(l_old, l_new):
    v_old = np.array(l_old['vecs'])
    v_new = np.array(l_new['vecs'])
    texts_new = l_new['texts']
    
    c_old = np.mean(v_old, axis=0).reshape(1, -1)
    
    # 1. Global Distance (The baseline)
    c_new_full = np.mean(v_new, axis=0).reshape(1, -1)
    global_distance = 1 - np.clip(cosine_similarity(c_old, c_new_full)[0][0], -1, 1)
    
    sentence_impacts = []
    
    # 2. Leave-One-Out (One by one)
    for j in range(len(v_new)):
        mask = np.ones(len(v_new), dtype=bool)
        mask[j] = False
        
        if np.any(mask):
            c_loo = np.mean(v_new[mask], axis=0).reshape(1, -1)
            dist_without_j = 1 - np.clip(cosine_similarity(c_old, c_loo)[0][0], -1, 1)
            # Impact is the reduction in distance
            impact = global_distance - dist_without_j
            sentence_impacts.append({'text': texts_new[j], 'impact': impact, 'index': j})
        else:
            sentence_impacts.append({'text': texts_new[j], 'impact': 0, 'index': j})

    # 3. Dynamic Filtering (The "Gap" Logic)
    # Sort by impact descending
    sorted_impacts = sorted(sentence_impacts, key=lambda x: x['impact'], reverse=True)
    
    # Filter for sentences that actually pull away (impact > 0)
    positive_impacts = [s for s in sorted_impacts if s['impact'] > 0]
    
    informers = []
    if positive_impacts:
        # We take the top influencer, then any others that are close to it
        # (within 50% of the top impact) to capture clusters/multiple senses
        max_imp = positive_impacts[0]['impact']
        informers = [s for s in positive_impacts if s['impact'] >= (max_imp * 0.5)]

    # 4. Group Impact (Persistent Shift Calculation)
    innovator_indices = [s['index'] for s in informers]
    mask_stable = np.ones(len(v_new), dtype=bool)
    
    if innovator_indices:
        mask_stable[innovator_indices] = False
    
    if np.any(mask_stable):
        c_stable = np.mean(v_new[mask_stable], axis=0).reshape(1, -1)
        persistent_shift = 1 - np.clip(cosine_similarity(c_old, c_stable)[0][0], -1, 1)
    else:
        # If all sentences were informers (Rare), shift is global
        persistent_shift = global_distance

    return {
        'global_distance': global_distance,
        'persistent_shift': persistent_shift,
        'informer_data': [(s['text'], s['impact']) for s in informers],
        'innovation_count': len(informers),
        'max_impact': max_imp if positive_impacts else 0
    }

In [13]:
def analyze_semantic_drift(df, target_lemma=None, old_century=14, new_century=20):
    # Standardize Century Column
    df['century_str'] = df['century'].astype(str)
    
    # 1. Filter and Map
    map_old = get_lemma_map(df[df['century_str'] == str(old_century)])
    map_new = get_lemma_map(df[df['century_str'] == str(new_century)])

    # 2. Execution Logic
    if target_lemma:
        if target_lemma in map_old and target_lemma in map_new:
            return run_multi_loo(map_old[target_lemma], map_new[target_lemma])
        return "Lemma not found in both centuries."
    else:
        all_results = []
        common = set(map_old.keys()) & set(map_new.keys())
        for l in common:
            # Check length of the vecs list to ensure LOO is possible
            if len(map_new[l]['vecs']) >= 2: 
                res = run_multi_loo(map_old[l], map_new[l])
                res['lemma'] = l
                all_results.append(res)
        
        return pd.DataFrame(all_results).sort_values('persistent_shift', ascending=False)

In [14]:
# results_df = analyze_semantic_drift(df_results, old_century=13, new_century=20)

In [15]:
# results_df.head(30)

In [16]:
# word_stats = analyze_semantic_drift(df_results, target_lemma='සතර', old_century=19, new_century=15)
# print(word_stats)

In [17]:
# word_stats

In [19]:
with open("../Data/consistent_keys_lemmas.json", "r", encoding="utf-8") as f:
    consistent_keys = json.load(f)

In [20]:
# # 1. Initialize an empty list to store the results
# all_lemma_stats = []

# # 2. Iterate through your target lemmas
# all_lemma_stats = []

# for lemma in consistent_keys:
#     word_stats = analyze_semantic_drift(df_results, target_lemma=lemma, old_century=15, new_century=19)
    
#     # Check if word_stats is actually a dictionary before assigning
#     if isinstance(word_stats, dict):
#         word_stats['lemma'] = lemma
#         all_lemma_stats.append(word_stats)
#     else:
#         print(f"Skipping lemma '{lemma}': Function returned {type(word_stats)} instead of dict.")

# analysis_df = pd.DataFrame(all_lemma_stats)

# # 4. Optional: Calculate the Impact Ratio for your thesis
# # This shows what % of the drift is explained by the innovations found
# analysis_df['drift_reduction_pct'] = (
#     (analysis_df['global_distance'] - analysis_df['persistent_shift']) / 
#     analysis_df['global_distance'] * 100
# ).fillna(0)

# # Reorder columns for better readability
# cols = ['lemma', 'global_distance', 'persistent_shift', 'drift_reduction_pct', 'innovation_count', 'informer_data']
# analysis_df = analysis_df[cols]

# # Display top drifted words
# print(analysis_df.sort_values(by='global_distance', ascending=False).head(10))

In [21]:
# 1. Configuration
centuries = [13, 14, 15, 18, 19, 20]
century_pairs = list(permutations(centuries, 2))
all_lag_stats = []

print(f"Processing {len(century_pairs)} permutations for {len(consistent_keys)} lemmas...")

# 2. Outer loop for Century Pairs
for old_c, new_c in tqdm(century_pairs, desc="Century Permutations"):
    
    perm_label = f"{old_c}_{new_c}"
    
    # Inner loop for Lemmas
    for lemma in consistent_keys:
        # We need the count of the target century instances
        # Assuming df_results is your source DataFrame
        target_instances = df_results[(df_results['lemma'] == lemma) & (df_results['century'] == new_c)]
        total_count = len(target_instances)
        
        # Call analysis
        word_stats = analyze_semantic_drift(
            df_results, 
            target_lemma=lemma, 
            old_century=old_c, 
            new_century=new_c
        )
        
        if isinstance(word_stats, dict):
            word_stats['lemma'] = lemma
            word_stats['permutation'] = perm_label
            word_stats['anchor_century'] = old_c
            word_stats['target_century'] = new_c
            word_stats['total_instance_count'] = total_count # New column added
            word_stats['direction'] = "Forward" if new_c > old_c else "Reverse"
            
            all_lag_stats.append(word_stats)

# 3. Create DataFrame
analysis_df = pd.DataFrame(all_lag_stats)

# 4. Final Processing
if not analysis_df.empty:
    # Handle percentages
    analysis_df['drift_reduction_pct'] = 0.0
    mask = (analysis_df['global_distance'] > 0)
    analysis_df.loc[mask, 'drift_reduction_pct'] = (
        (analysis_df['global_distance'] - analysis_df['persistent_shift']) / 
        analysis_df['global_distance'] * 100
    )

    # 5. Column Organization
    cols = [
        'lemma', 'permutation', 'anchor_century', 'target_century', 
        'direction', 'global_distance', 'persistent_shift', 
        'drift_reduction_pct', 'innovation_count', 'total_instance_count', 'informer_data'
    ]
    analysis_df = analysis_df[[c for c in cols if c in analysis_df.columns]]

    print(f"\nSuccess: {len(analysis_df)} rows generated.")

Processing 30 permutations for 261 lemmas...


Century Permutations: 100%|██████████| 30/30 [43:44<00:00, 87.48s/it] 


Success: 7830 rows generated.


In [22]:
analysis_df.sort_values(by='drift_reduction_pct', ascending=False)

,lemma,permutation,anchor_century,target_century,direction,global_distance,persistent_shift,drift_reduction_pct,innovation_count,total_instance_count,informer_data
4536,කෙසේද,18_15,18,15,Reverse,0.001761,1.192093e-07,99.993225,1,2,[(සුලකළ අන්ය ලකෘත වූ හෙළ රුවනින් ස්ඵටික පලිඟු ...
1498,ම,14_13,14,13,Reverse,0.002965,7.443428e-04,74.893951,35,66,[(විසුද්ධි යයි සියලු රාගාදි මලයෙන් වෙන් වූ සර්...
4337,ලද්ද,18_14,18,14,Reverse,0.009036,2.305031e-03,74.491943,1,2,[(තවද මේ අනාගතවංශ දේශනාව කවරෙකු විසින් දෙසන ලද...
6647,ආදි,20_13,20,13,Reverse,0.003974,1.034737e-03,73.959351,8,22,[(සලක ස්ථානයෙහි හෙවත් ගණය මුලුල්ලම ගුරු වුව හො...
4484,යත්,18_15,18,15,Reverse,0.004249,1.143217e-03,73.097313,4,7,[(බියකරු සසර සසුරේ භයංකර සංසාර සාගරයෙහි බමණ භ්...
5342,ආදි,19_13,19,13,Reverse,0.005155,1.446486e-03,71.937698,8,22,[(සලක ස්ථානයෙහි හෙවත් ගණය මුලුල්ලම ගුරු වුව හො...
977,පාද,13_19,13,19,Forward,0.003983,1.124501e-03,71.770164,5,14,[(යනු හෙයින් ප්රස්තාරය නෂ්ටප්රත්යය උද්දිෂ්ටය එ...
3629,ප්රකාර,15_19,15,19,Forward,0.007322,2.122998e-03,71.005974,2,4,[(මේ උක්ත ප්රකාර නාමයෙන් පිටු කින් යුක්ත කොට ස...
5438,රාජ,19_13,19,13,Reverse,0.002874,8.625984e-04,69.982574,8,15,[(ඒ වටා ස්වස්තිකය ශ්රීවත්සය නන්දාවර්තය සෞවස්ති...
5146,ය,18_20,18,20,Forward,0.001799,5.486012e-04,69.507019,102,260,"[(කළු කළ යනු බස් ආදි බව හඟවයි, 1.8060207e-05),..."


In [23]:
# analysis_df = analysis_df.sort_values(by='drift_reduction_pct', ascending=False)

In [24]:
analysis_df.to_csv("../Data/XLM-R_lemma_drift_analysis.csv", index=False, encoding='utf-8-sig')